In [59]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


In [42]:
import pandas as pd
import os
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)


from config.constants import GENERATION_MODELS,SLEEP_BETWEEN_CALLS,VALID_CATEGORIES ,DATA_PATH



MODELS_TO_RUN = [
    "gpt54_mini",
    "gpt4o_mini",
    "gpt_codex",
    "gpt_sol",
    # "gpt_oss"
    # "gpt_luna",
]


# MAX_WORKERS = len(MODELS_TO_RUN)

OPENAI_API_KEY = os.getenv(
    "OPENAI_API_KEY"
)
HUGGING_FACE_API_KEY = os.getenv(
    "HUGGING_FACE_API_KEY"
)

openai_client = OpenAI(
    api_key=OPENAI_API_KEY
)

huggingface_client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HUGGING_FACE_API_KEY,
)

df=pd.read_csv(f'{DATA_PATH}df_n2.csv')
df=df[400:].copy()

In [43]:
import json
import time
import uuid
import random
import pandas as pd



def create_usage_tracker():
    return {
        "prompt_tokens": 0,
        "reasoning_tokens": 0,
        "total_tokens": 0,
        "num_calls": 0,
    }


def update_usage(usage_tracker, usage):
    """
    Accumulate token usage for one model.
    """

    if usage is None:
        return

    u = usage.model_dump()

    usage_tracker["prompt_tokens"] += u.get(
        "prompt_tokens", 0
    )

    usage_tracker["total_tokens"] += u.get(
        "total_tokens", 0
    )

    usage_tracker["reasoning_tokens"] += (
        u.get("completion_tokens_details", {})
         .get("reasoning_tokens", 0)
    )

    usage_tracker["num_calls"] += 1



def build_messages(row):

    system_prompt = f"""
You are an expert software engineer performing a professional code review.

INPUT:
1. Pull request context (when available)
2. Surrounding code context (when available), wrapped in <context> tags
3. The code patch (git diff), wrapped in <patch> tags

TASK:
Identify review comments that a human reviewer would reasonably leave on the changes in <patch>. For each category below, use the available context when it helps determine whether the patch introduces an issue or an opportunity for improvement:

- CORRECTNESS: Does the patch introduce an error, undefined reference, or unexpected behavior? Does a changed name/parameter/value conflict with, or fail to match, how it's used elsewhere in <context>? Point to the specific location if so.
- MAINTAINABILITY: Does the patch duplicate, remove, or diverge from logic that exists in a structurally similar place in <context> (sibling method, sibling class, repeated pattern)? Is there hard-coded or fragile logic that will make future changes harder? Prefer suggesting consolidation over just noting duplication.
- DESIGN: Is there a simpler, more consistent architecture or implementation pattern that fits how similar problems are solved elsewhere in <context>?
- READABILITY: Is the code harder to understand than it needs to be, independent of whether it's correct — unclear naming, formatting, or control flow?
- DOCUMENTATION: Is a docstring or comment now inaccurate, missing, or inconsistent with the code's actual behavior? If a name/value conflicts with what's documented elsewhere in <context> without affecting runtime behavior, treat it as Documentation rather than Correctness.

RULES:
- Only comment on things caused by or directly connected to <patch>. Don't flag unrelated pre-existing issues.
- Don't hallucinate code that isn't shown.
- One issue per comment, and each comment belongs to exactly one category. Be concise and specific — point at the exact location/identifier rather than restating the diff.
- It's fine to phrase a comment as a direct question when that's how a reviewer would naturally raise it.
- Before assigning a category, briefly justify why that category fits over the others it could plausibly be confused with (e.g. Design vs. Readability, Correctness vs. Documentation).

Categories:
{VALID_CATEGORIES}

OUTPUT (STRICT JSON ONLY, no other text):
{{"comments": [{{"comment": "...", "category_justification": "...", "category": "...", "severity": "Low | Medium | High"}}]}}

If nothing meaningful:
{{"comments": []}}
"""

    user_prompt = ""

    if "pr_title" in row and pd.notna(row["pr_title"]):

        user_prompt += f"""
PULL REQUEST TITLE:
{row["pr_title"]}

"""

    if "target_file" in row and pd.notna(row["target_file"]):

        user_prompt += f"""
TARGET FILE:
{row["target_file"]}

"""

    if (
        "relevant_same_file_code_hunks" in row
        and pd.notna(row["relevant_same_file_code_hunks"])
    ):

        user_prompt += f"""
RELATED CODE HUNKS IN TARGET FILE:
{row["relevant_same_file_code_hunks"]}

"""

    if "changed_files" in row and pd.notna(row["changed_files"]):

        user_prompt += f"""
FILES CHANGED:
{row["changed_files"]}

"""

    if (
        "relevant_different_files_code_hunks" in row
        and pd.notna(row["relevant_different_files_code_hunks"])
    ):

        user_prompt += f"""
RELATED CODE HUNKS IN DIFFERENT FILE:
{row["relevant_different_files_code_hunks"]}

"""

    if "relevant_context" in row and pd.notna(row["relevant_context"]):

        if str(row["relevant_context"]).strip():

            user_prompt += """
SURROUNDING CODE CONTEXT:
This is additional context extracted from the original file around the changed code to help you in problem identification.

<context>
"""

            user_prompt += str(row["relevant_context"])

            user_prompt += """
</context>

"""

    user_prompt += f"""
CODE PATCH:
<patch>
{row["hunk"]}
</patch>
"""

    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        },
    ]


# ============================================================
# Output parsing
# ============================================================

def parse_output(text, code_diff_idx):

    if text is None:
        return []

    try:

        data = json.loads(text)

        comments = data.get("comments", [])

        parsed = []

        for j, c in enumerate(comments):

            category = str(
                c.get("category", "")
            ).strip()

            comment = str(
                c.get("comment", "")
            ).strip()

            severity = str(
                c.get("severity", "Medium")
            ).strip()

            if category not in VALID_CATEGORIES:
                continue

            if len(comment) == 0:
                continue

            parsed.append({
                "category": category,
                "comment": comment,
                "severity": severity,
            })

        return parsed

    except Exception:

        return []


def predict_once(
    row,
    model_config,
    usage_tracker
):

    messages = build_messages(row)
    API_TYPE= model_config["api"]  
    MODEL_NAME = model_config["name"]  
    REASONING_EFFORT=model_config.get("reasoning_effort")    

    if API_TYPE == "responses":

        kwargs = {
            "model": MODEL_NAME,
            "input": messages,
            "timeout": 120,
        }

        if REASONING_EFFORT:
            kwargs["reasoning"] = {
                "effort": REASONING_EFFORT
            }
        print(f"Calling Responses API codex with kwargs: {kwargs}")    

        response = openai_client.responses.create(**kwargs)

        raw_text = response.output_text

    else:

        kwargs = {
            "model": MODEL_NAME,
            "messages": messages,
            "timeout": 120,
        }


        if REASONING_EFFORT:
            kwargs["reasoning_effort"] = REASONING_EFFORT
            # Select the appropriate client
        if model_config["provider"] == "huggingface":
            client = huggingface_client
        else:
            client = openai_client
            

        response = client.chat.completions.create(
            **kwargs
        )

        raw_text = response.choices[0].message.content


    comments = parse_output(
        raw_text,
        row["idx"] if "idx" in row else None
    )

    for comment in comments:

        comment["id"] = str(uuid.uuid4())

        comment["generation_system"] = (
            model_config["name"]
        )

    update_usage(
        usage_tracker,
        response.usage
    )

    return comments


# ============================================================
# Pipeline execution
# ============================================================

def run_pipeline(df, model_config):
    # Each model has its own usage tracker
    usage_tracker = create_usage_tracker()

    preds = []

    total = len(df)
    processed = 0

    base_sleep = (
        SLEEP_BETWEEN_CALLS
        if "SLEEP_BETWEEN_CALLS" in globals()
        else 1.5
    )

    model_name = model_config["name"]

    print(
        f"\n[START] {model_name} "
        f"- Processing {total} rows"
    )

    for _, row in df.iterrows():

        processed += 1

        print(
            f"\n[{model_name}] "
            f"[ROW {processed}/{total}] Starting"
        )

        # ----------------------------------------------------
        # Empty hunk
        # ----------------------------------------------------

        if pd.isna(row["hunk"]):

            preds.append([])

            continue

        success = False
        sleep_time = base_sleep

        # ----------------------------------------------------
        # Retry
        # ----------------------------------------------------

        for attempt in range(3):

            try:

                comments = predict_once(
                    row,
                    model_config,
                    usage_tracker
                )

                preds.append(comments)

                success = True

                break

            except Exception as e:

                print(
                    f"[{model_name}] "
                    f"[ERROR] Attempt {attempt + 1}/3: {e}"
                )

                wait = (
                    sleep_time
                    + random.uniform(0, 1)
                )

                time.sleep(wait)

                sleep_time *= 2

        # ----------------------------------------------------
        # Failed after retries
        # ----------------------------------------------------

        if not success:

            preds.append([])

        # ----------------------------------------------------
        # Delay between API calls
        # ----------------------------------------------------

        wait = (
            base_sleep
            + random.uniform(0, 0.8)
        )

        time.sleep(wait)

        # ----------------------------------------------------
        # Progress
        # ----------------------------------------------------

        if processed % 10 == 0:

            print(
                f"[{model_name}] "
                f"[CHECKPOINT] "
                f"{processed}/{total}"
            )

    # ========================================================
    # Add generated comments to dataframe
    # ========================================================

    df = df.reset_index(drop=True)

    df["generated_comments"] = preds

    print(
        f"\n[FINISHED] {model_name}"
    )

    print(
        f"[USAGE] "
        f"Prompt: {usage_tracker['prompt_tokens']} | "
        f"Reasoning: {usage_tracker['reasoning_tokens']} | "
        f"Total: {usage_tracker['total_tokens']} | "
        f"Calls: {usage_tracker['num_calls']}"
    )

    return df, usage_tracker

In [39]:
import pandas as pd

def create_generated_comments_dataset(df):
    rows = []

    for _, row in df.iterrows():
        comments = row["generated_comments"]

        # Skip empty/null comments
        if not isinstance(comments, list):
            continue

        for comment in comments:
            new_row = {
                "comment_id": comment.get("id"),
                "comment": comment.get("comment"),
                "category": comment.get("category"),
                "generation_system": comment.get("generation_system"),
                "patch_id": row["patch_id"],
                "severity": comment.get("severity"),
            }

            rows.append(new_row)

    return pd.DataFrame(rows)

In [44]:
import os
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

from dotenv import load_dotenv

load_dotenv(override=True)



def run_model(model_number, model_key):

    model_config = GENERATION_MODELS[model_key]

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"MODEL {model_number}: {model_config}"
    )


    results_df, usage = run_pipeline(
        df.copy(),
        model_config
    )



    generated_df = create_generated_comments_dataset(
        results_df
    )

    generated_df.to_csv(f"{DATA_PATH}df_n3_5_{model_number}.csv", index=False)


    print('end=============================== model execution finished')

    return {
        "model_number": model_number,
        "model_key": model_key,
        "model_name": model_config["name"],
        "usage": usage,
    }



def run_all_models(
    models_to_run,
):
    max_workers = len(models_to_run)
    print(
        f"[INFO] Running {max_workers} max workers for {len(models_to_run)} models ")

    results = {}

    print(
        f"[START] Running {len(models_to_run)} models "
        f"in parallel"
    )

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        futures = {}

        for model_number, model_key in enumerate(
            models_to_run,
            start=1
        ):

            future = executor.submit(
                run_model,
                model_number,
                model_key,
            )

            futures[future] = (
                model_number,
                model_key
            )

        # Collect results as models finish
        for future in as_completed(futures):

            model_number, model_key = futures[future]

            try:

                result = future.result()

                results[model_number] = result

                # Save token usage for this model
                save_token_usage_log(
                    model_name=result["model_name"],
                    usage=result["usage"]
                )

                print(
                    f"[COMPLETED] "
                    f"Model {model_number}: {model_key}"
                )

            except Exception as e:

                print(
                    f"[FAILED] "
                    f"Model {model_number}: {model_key}"
                )

                print(
                    f"[ERROR] {e}"
                )

    # ========================================================
    # Final summary
    # ========================================================

    print("\n[SUMMARY]")

    for model_number, model_key in enumerate(
        models_to_run,
        start=1
    ):

        if model_number in results:

            result = results[model_number]

            print(
                f"[OK] {model_number} - {model_key} "
            )

        else:

            print(
                f"[FAILED] {model_number} - {model_key}"
            )

    return results

In [45]:
results=run_all_models(
    MODELS_TO_RUN,
)

[INFO] Running 4 max workers for 4 models 
[START] Running 4 models in parallel

MODEL 1: {'name': 'gpt-5.4-mini-2026-03-17', 'reasoning_effort': 'low', 'provider': 'openai', 'api': 'chat'}

[START] gpt-5.4-mini-2026-03-17 - Processing 105 rows

[gpt-5.4-mini-2026-03-17] [ROW 1/105] Starting

MODEL 2: {'name': 'gpt-4o-mini-2024-07-18', 'reasoning_effort': '', 'provider': 'openai', 'api': 'chat'}

[START] gpt-4o-mini-2024-07-18 - Processing 105 rows

[gpt-4o-mini-2024-07-18] [ROW 1/105] Starting

MODEL 3: {'name': 'gpt-5.3-codex', 'reasoning_effort': 'low', 'provider': 'openai', 'api': 'responses'}

[START] gpt-5.3-codex - Processing 105 rows

[gpt-5.3-codex] [ROW 1/105] Starting
Calling Responses API codex with kwargs: {'model': 'gpt-5.3-codex', 'input': [{'role': 'system', 'content': '\nYou are an expert software engineer performing a professional code review.\n\nINPUT:\n1. Pull request context (when available)\n2. Surrounding code context (when available), wrapped in <context> tags\n

In [8]:
import json
from datetime import datetime
from pathlib import Path


def save_token_usage_log(
    model_name,
    usage,
    task_name="comment_generation",
    log_file="../logs/token_usage_insights_logs.json"
):

    usage_stats = {
        "task": task_name,
        "timestamp": datetime.now().isoformat(),
        "model_name": model_name,
        "num_calls": usage["num_calls"],
        "prompt_tokens": usage["prompt_tokens"],
        "reasoning_tokens": usage["reasoning_tokens"],
        "total_tokens": usage["total_tokens"],
    }

    log_path = Path(log_file)

    log_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        with open(log_path, "r") as f:
            logs = json.load(f)

    except (FileNotFoundError, json.JSONDecodeError):
        logs = []

    logs.append(usage_stats)

    with open(log_path, "w") as f:
        json.dump(
            logs,
            f,
            indent=4
        )

    print(f"Token usage saved to {log_path}")

In [9]:
df3=pd.read_csv('../data/df_n3_1.csv')

In [11]:
df3.head()

,comment_id,comment,category,generation_system,patch_id,severity
0,c2d17431-8926-4f9f-8821-9afc6b98b461,The new `vc3-builder` branch allocates `cmd` w...,Correctness,gpt-5.4-mini-2026-03-17,P000001,Low
1,bba8859e-da1f-44af-8cf2-a604e096696f,"At `initProvider`, `shouldComponentUpdate` now...",Correctness,gpt-5.4-mini-2026-03-17,P000004,High


In [17]:
import json
from datetime import datetime
from pathlib import Path


def save_token_usage_log(
    task_name="comment_generation",
    log_file="../logs/token_usage_insights_logs.json"
):

    usage_stats = {
        "task": task_name,
        "timestamp": datetime.now().isoformat(),
        "model_name": MODEL_NAME,
        "num_calls": TOTAL_USAGE["num_calls"],
        "prompt_tokens": TOTAL_USAGE["prompt_tokens"],
        "reasoning_tokens": TOTAL_USAGE["reasoning_tokens"],
        "total_tokens": TOTAL_USAGE["total_tokens"],
    }

    log_path = Path(log_file)

    log_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    try:
        with open(log_path, "r") as f:
            logs = json.load(f)

    except (FileNotFoundError, json.JSONDecodeError):
        logs = []

    logs.append(usage_stats)

    with open(log_path, "w") as f:
        json.dump(
            logs,
            f,
            indent=4
        )

    print(f"Token usage saved to {log_path}")
    
save_token_usage_log()    

Token usage saved to ..\logs\token_usage_insights_logs.json


In [ ]:
from concurrent.futures import ThreadPoolExecutor


def run_all_models(df, models):

    with ThreadPoolExecutor(
        max_workers=len(models)
    ) as executor:

        futures = {
            executor.submit(
                run_pipeline,
                df,
                model_config
            ): model_config["name"]
            for model_config in models
        }

        results = {}

        for future, model_name in futures.items():

            try:
                results[model_name] = future.result()

            except Exception as e:
                print(
                    f"[ERROR] Model {model_name} failed: {e}"
                )

    return results

In [ ]:
results = run_all_models(
    df,
    GENERATION_MODELS
)

creating the jsonl file for server execution

In [32]:
import json
import pandas as pd
from config.constants import VALID_CATEGORIES

def has_content(value):
    if pd.isna(value):
        return False

    if isinstance(value, (list, tuple, set)):
        return len(value) > 0

    if isinstance(value, str):
        return value.strip() not in ("", "[]", "{}")

    return True

def build_messages(row):

    categories_text = "\n".join(
        f"- {category}"
        for category in VALID_CATEGORIES
    )

    system_prompt = f"""
You are an expert software engineer performing a professional code review.

INPUT:
The following information may be provided:
1. Pull request title, wrapped in <pr_title> tags
2. Target file, wrapped in <target_file> tags
3. Related code hunks from the target file, wrapped in <same_file_code_hunks> tags
4. Files changed in the pull request, wrapped in <changed_files> tags
5. Related code hunks from different files, wrapped in <different_file_code_hunks> tags
6. Surrounding code context, wrapped in <context> tags
7. The code patch (git diff), wrapped in <patch> tags

TASK:
Identify review comments that a human reviewer would reasonably leave on the changes in <patch>. For each category below, use the available context when it helps determine whether the patch introduces an issue or an opportunity for improvement:

- CORRECTNESS: Does the patch introduce an error, undefined reference, or unexpected behavior? Does a changed name/parameter/value conflict with, or fail to match, how it's used elsewhere in <context>? Point to the specific location if so.
- MAINTAINABILITY: Does the patch duplicate, remove, or diverge from logic that exists in a structurally similar place in <context> (sibling method, sibling class, repeated pattern)? Is there hard-coded or fragile logic that will make future changes harder? Prefer suggesting consolidation over just noting duplication.
- DESIGN: Is there a simpler, more consistent architecture or implementation pattern that fits how similar problems are solved elsewhere in <context>?
- READABILITY: Is the code harder to understand than it needs to be, independent of whether it's correct — unclear naming, formatting, or control flow?
- DOCUMENTATION: Is a docstring or comment now inaccurate, missing, or inconsistent with the code's actual behavior? If a name/value conflicts with what's documented elsewhere in <context> without affecting runtime behavior, treat it as Documentation rather than Correctness.

RULES:
- Only comment on things caused by or directly connected to <patch>. Don't flag unrelated pre-existing issues.
- Don't hallucinate code that isn't shown.
- One issue per comment, and each comment belongs to exactly one category.
- Be concise and specific. Point to the exact location or identifier rather than restating the diff.
- It's fine to phrase a comment as a direct question when that's how a reviewer would naturally raise it.
- Before assigning a category, briefly justify why that category fits over the others it could plausibly be confused with (e.g. Design vs. Readability, Correctness vs. Documentation).
- Treat the tagged inputs as separate sources of information. Use them only when relevant to understanding the changes in <patch>.

Categories:
{categories_text}

OUTPUT (STRICT JSON ONLY, no other text):
{{"comments": [{{"comment": "...", "category_justification": "...", "category": "...", "severity": "Low | Medium | High"}}]}}

If nothing meaningful:
{{"comments": []}}
"""
    user_prompt = ""

    if "pr_title" in row and has_content(row["pr_title"]):
        user_prompt += f"""
    <pr_title>
    {row["pr_title"]}
    </pr_title>

    """

    if "target_file" in row and has_content(row["target_file"]):
        user_prompt += f"""
    <target_file>
    {row["target_file"]}
    </target_file>

    """

    if (
        "relevant_same_file_code_hunks" in row
        and has_content(row["relevant_same_file_code_hunks"])
    ):
        user_prompt += f"""
    <same_file_code_hunks>
    {row["relevant_same_file_code_hunks"]}
    </same_file_code_hunks>

    """

    if "changed_files" in row and has_content(row["changed_files"]):
        user_prompt += f"""
    <changed_files>
    {row["changed_files"]}
    </changed_files>

    """

    if (
        "relevant_different_files_code_hunks" in row
        and has_content(row["relevant_different_files_code_hunks"])
    ):
        user_prompt += f"""
    <different_file_code_hunks>
    {row["relevant_different_files_code_hunks"]}
    </different_file_code_hunks>

    """

    if "relevant_context" in row and has_content(row["relevant_context"]):
        user_prompt += f"""
    <context>
    {row["relevant_context"]}
    </context>

    """

    if "hunk" in row and has_content(row["hunk"]):
        user_prompt += f"""
    <patch>
    {row["hunk"]}
    </patch>
    """


    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]


def generate_prompt_jsonl(df, output_file):
    """
    Generate a JSONL file containing all prompts.

    One line = one model request for one patch.

    No API calls are made.
    """

    total_prompts = 0

    with open(output_file, "w", encoding="utf-8") as f:

        for _, row in df.iterrows():

            if pd.isna(row["hunk"]):
                continue

            messages = build_messages(row)

            request = {
                "patch_id": row["patch_id"]
                if "patch_id" in row
                else None,
                "messages": messages
            }

            f.write(
                json.dumps(
                    request,
                    ensure_ascii=False
                ) + "\n"
            )

            total_prompts += 1

    print(f"\n[DONE] Generated {total_prompts} prompts")
    print(f"[OUTPUT] {output_file}")

In [33]:
df_n2=pd.read_csv('../data/df_n2.csv')

In [34]:
generate_prompt_jsonl(
    df_n2,
    output_file="../data/df_n3_prompts_500.jsonl"
)


[DONE] Generated 500 prompts
[OUTPUT] ../data/df_n3_prompts_500.jsonl


In [37]:
df_n2.head(20)

,old_hunk,oldf,hunk,comment,ids,repo,ghid,old,new,lang,...,pr_title,changed_files,target_file,num_code_hunks,pr_code_hunks,same_file_code_hunks,code_hunks_num_same_file,relevant_context,relevant_different_files_code_hunks,relevant_same_file_code_hunks
0,"@@ -273,6 +273,10 @@ class RootPathHandler(Bas...",# -*- coding: utf-8 -*-\n#\n# Copyright 2012-2...,"@@ -274,6 +274,7 @@ class RootPathHandler(Base...","Is the name ""head"" a convention for health che...","[19459, 'f9d15209195d2cb49052b3662bbe21e387d1a...",spotify/luigi,2789,"self.redirect(""/static/visualiser/ind...","self.redirect(""/static/visualiser/ind...",py,...,Add HEAD endpoint to scheduler server for stat...,"['luigi/server.py', 'test/server_test.py']",luigi/server.py,2,"[{'filename': 'luigi/server.py', 'patch': '@@ ...","[{'filename': 'luigi/server.py', 'patch': '@@ ...",1,class RootPathHandler(BaseTaskHistoryHandler):...,"[{'filename': 'test/server_test.py', 'patch': ...","[{'filename': 'luigi/server.py', 'patch': '@@ ..."
1,"@@ -117,8 +117,7 @@ def main(args):\n ...","""""""\nModeling Relational Data with Graph Convo...","@@ -117,7 +117,8 @@ def main(args):\n ...","`F.cross_entropy`? Also, isn't `tran_acc` requ...","[36336, '3fe1aee96f1f3dd5359b6bf65f387f8e45c6a...",dmlc/dgl,1217,backward_time.append(t2 - t1)\n ...,backward_time.append(t2 - t1)\n ...,py,...,[Bugfix] Correct the loss function in RGCN model,"['examples/mxnet/rgcn/entity_classify.py', 'ex...",examples/mxnet/rgcn/entity_classify.py,3,[{'filename': 'examples/mxnet/rgcn/entity_clas...,[{'filename': 'examples/mxnet/rgcn/entity_clas...,1,def main(args):\n # split dataset into trai...,[],[{'filename': 'examples/mxnet/rgcn/entity_clas...
2,"@@ -48,15 +48,20 @@ class ConsoleLineEdit(misc...",# vim: ft=python fileencoding=utf-8 sts=4 sw=4...,"@@ -48,12 +48,6 @@ class ConsoleLineEdit(miscw...",Couldn't you move the stylesheet to `ConsoleWi...,"[25004, '29887dfc607c94dcaad268f5b64de99e01a76...",qutebrowser/qutebrowser,5515,execute = pyqtSignal(str)\n- STYLESHEE...,execute = pyqtSignal(str)\n def __ini...,py,...,Remove QtFont as a config type,"['doc/help/settings.asciidoc', 'qutebrowser/co...",qutebrowser/misc/consolewidget.py,8,"[{'filename': 'doc/help/settings.asciidoc', 'p...",[{'filename': 'qutebrowser/misc/consolewidget....,6,class ConsoleLineEdit(miscwidgets.CommandLineE...,"[{'filename': 'doc/help/settings.asciidoc', 'p...",[{'filename': 'qutebrowser/misc/consolewidget....
3,"@@ -2228,6 +2228,94 @@ def iter_kms_keyrings(s...",# Copyright 2017 The Forseti Security Authors....,"@@ -2229,7 +2229,7 @@ class ApiClientImpl(ApiC...",please update all the docstring below in other...,"[34229, '57d873c86c7cf552c73f82a1636cfd46097bb...",forseti-security/forseti-security,2916,'this API ...,'this API ...,py,...,Added kubernetes resources from CAI,"['configs/server/forseti_conf_server.yaml.in',...",google/cloud/forseti/services/inventory/base/g...,10,[{'filename': 'configs/server/forseti_conf_ser...,[{'filename': 'google/cloud/forseti/services/i...,2,class ApiClientImpl(ApiClient):\n def iter_...,[{'filename': 'google/cloud/forseti/services/i...,[{'filename': 'google/cloud/forseti/services/i...
4,"@@ -45,18 +45,61 @@ def dice_loss(pred,\n ...",# Copyright (c) OpenMMLab. All rights reserved...,"@@ -51,8 +51,8 @@ def naive_dice_loss(pred,\n ...",use_second_power=True -> naive_dice=False,"[26623, '84fbc37749351e8be21733a135e41823c594e...",open-mmlab/mmdetection,6607,"eps=1e-3,\n ...","eps=1e-3,\n ...",py,...,[feature] add MaskFormer to mmdet,"['configs/maskformer/README.md', 'configs/mask...",mmdet/models/losses/dice_loss.py,22,"[{'filename': 'configs/maskformer/README.md', ...",[{'filename': 'mmdet/models/losses/dice_loss.p...,6,"def naive_dice_loss(pred,\n ...",[{'filename': 'configs/maskformer/maskformer_r...,[{'filename': 'mmdet/models/losses/dice_loss.p...
5,"@@ -903,6 +903,10 @@ def write(self, obj):\n ...",# -*- Mode: python; tab-width: 4; indent-tabs-...,"@@ -903,10 +903,6 @@ class PDBWriter(base.Writ...",```sugg

In [35]:
import json

with open("../data/df_n3_prompts_500.jsonl", "r", encoding="utf-8") as f:
    first_line = json.loads(f.readline())

print(first_line)

{'patch_id': 'P000000', 'messages': [{'role': 'system', 'content': '\nYou are an expert software engineer performing a professional code review.\n\nINPUT:\nThe following information may be provided:\n1. Pull request title, wrapped in <pr_title> tags\n2. Target file, wrapped in <target_file> tags\n3. Related code hunks from the target file, wrapped in <same_file_code_hunks> tags\n4. Files changed in the pull request, wrapped in <changed_files> tags\n5. Related code hunks from different files, wrapped in <different_file_code_hunks> tags\n6. Surrounding code context, wrapped in <context> tags\n7. The code patch (git diff), wrapped in <patch> tags\n\nTASK:\nIdentify review comments that a human reviewer would reasonably leave on the changes in <patch>. For each category below, use the available context when it helps determine whether the patch introduces an issue or an opportunity for improvement:\n\n- CORRECTNESS: Does the patch introduce an error, undefined reference, or unexpected behav

In [39]:
df_n2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 22 columns):
 #   Column                               Non-Null Count  Dtype 
---  ------                               --------------  ----- 
 0   old_hunk                             500 non-null    object
 1   oldf                                 500 non-null    object
 2   hunk                                 500 non-null    object
 3   comment                              500 non-null    object
 4   ids                                  500 non-null    object
 5   repo                                 500 non-null    object
 6   ghid                                 500 non-null    int64 
 7   old                                  500 non-null    object
 8   new                                  500 non-null    object
 9   lang                                 500 non-null    object
 10  patch_id                             500 non-null    object
 11  pr_number                            500 non-

In [38]:
df_n2=pd.read_csv('../data/df_n2.csv')

In [47]:
df_n2[:100]['relevant_different_files_code_hunks'].head()

0    [{'filename': 'test/server_test.py', 'patch': ...
1                                                   []
2    [{'filename': 'doc/help/settings.asciidoc', 'p...
3    [{'filename': 'google/cloud/forseti/services/i...
4    [{'filename': 'configs/maskformer/maskformer_r...
Name: relevant_different_files_code_hunks, dtype: object

In [48]:
df=df_n2.copy()

In [52]:
df["relevant_different_files_code_hunks"].astype(str).str.strip().eq("[]").head(20)

0     False
1      True
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9      True
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
Name: relevant_different_files_code_hunks, dtype: bool

In [57]:
df["relevant_different_files_code_hunks"][380:].astype(str).str.strip().eq("[]").head(20)

380     True
381     True
382     True
383     True
384     True
385     True
386     True
387     True
388     True
389     True
390     True
391     True
392     True
393    False
394    False
395    False
396    False
397     True
398    False
399    False
Name: relevant_different_files_code_hunks, dtype: bool

In [60]:
df[:100]['relevant_same_file_code_hunks'].head()

0                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       